# Cost Analysis for Model Optimization

In this notebook, we'll analyze the cost implications of model optimization techniques for large production models. We'll explore how techniques like quantization, pruning, and knowledge distillation can reduce costs and improve ROI when deploying large language models (LLMs) and other deep learning models in production.

## 1. Import Dependencies

In [ ]:
# Import common modules
from common_imports import *

# Import specific modules for this notebook
from common_imports import *
from IPython.display import display, HTML, Markdown
import math

## 2. Load Metrics from Previous Notebooks

We'll load the metrics from our previous optimization experiments to understand the impact of each technique.

In [ ]:
# Load metrics from previous notebooks
try:
    with open('quantized-metrics.json', 'r') as f:
        quantized_metrics = json.load(f)
    print(f"Loaded quantized metrics for {len(quantized_metrics)} models")
except FileNotFoundError:
    print("quantized-metrics.json not found. Using empty dictionary.")
    quantized_metrics = {}

try:
    with open('pruned-metrics.json', 'r') as f:
        pruned_metrics = json.load(f)
    print(f"Loaded pruned metrics for {len(pruned_metrics)} models")
except FileNotFoundError:
    print("pruned-metrics.json not found. Using empty dictionary.")
    pruned_metrics = {}

try:
    with open('distilled-metrics.json', 'r') as f:
        distilled_metrics = json.load(f)
    print(f"Loaded distilled metrics for {len(distilled_metrics)} models")
except FileNotFoundError:
    print("distilled-metrics.json not found. Using empty dictionary.")
    distilled_metrics = {}

## 3. Define Production Model Scenarios

In production, we often work with much larger models than those in our workshop examples. Let's define some realistic production model scenarios to analyze cost implications.

In [ ]:
# Define production model scenarios
production_models = {
    "large_llm": {
        "name": "Large Language Model (175B parameters)",
        "size_gb": 350,  # Model size in GB
        "inference_time_ms": 500,  # Inference time in ms per request
        "instance_type": "ml.p4d.24xlarge",  # High-end GPU instance
        "memory_gb": 320,  # Memory usage in GB
        "batch_size": 8,  # Batch size for inference
        "throughput": 16  # Requests per second
    },
    "medium_llm": {
        "name": "Medium Language Model (30B parameters)",
        "size_gb": 60,  # Model size in GB
        "inference_time_ms": 150,  # Inference time in ms per request
        "instance_type": "ml.g5.12xlarge",  # Medium GPU instance
        "memory_gb": 45,  # Memory usage in GB
        "batch_size": 16,  # Batch size for inference
        "throughput": 64  # Requests per second
    },
    "vision_transformer": {
        "name": "Vision Transformer (ViT-L)",
        "size_gb": 1.5,  # Model size in GB
        "inference_time_ms": 80,  # Inference time in ms per request
        "instance_type": "ml.g5.2xlarge",  # Smaller GPU instance
        "memory_gb": 8,  # Memory usage in GB
        "batch_size": 32,  # Batch size for inference
        "throughput": 400  # Requests per second
    }
}

# Display production model scenarios
pd.DataFrame(production_models).T

## 4. Define AWS Instance Types and Costs

Let's define the costs for various AWS instance types, including standard GPU instances and specialized Amazon silicon options.

In [ ]:
# Define AWS instance types and costs
instance_costs = {
    # Standard GPU instances
    "ml.p4d.24xlarge": 32.77,    # $ per hour - 8 A100 GPUs
    "ml.p4de.24xlarge": 34.40,   # $ per hour - 8 A100 GPUs with more memory
    "ml.p5.48xlarge": 98.326,    # $ per hour - 8 H100 GPUs
    "ml.g5.xlarge": 1.006,       # $ per hour - 1 A10G GPU
    "ml.g5.2xlarge": 1.515,      # $ per hour - 1 A10G GPU
    "ml.g5.4xlarge": 2.176,      # $ per hour - 1 A10G GPU
    "ml.g5.8xlarge": 4.352,      # $ per hour - 1 A10G GPU
    "ml.g5.12xlarge": 6.528,     # $ per hour - 4 A10G GPUs
    "ml.g5.16xlarge": 8.704,     # $ per hour - 1 A10G GPU
    "ml.g5.24xlarge": 13.056,    # $ per hour - 4 A10G GPUs
    "ml.g5.48xlarge": 26.112,    # $ per hour - 8 A10G GPUs
    "ml.g4dn.xlarge": 0.736,     # $ per hour - 1 T4 GPU
    "ml.g4dn.2xlarge": 0.94,     # $ per hour - 1 T4 GPU
    "ml.g4dn.4xlarge": 1.505,    # $ per hour - 1 T4 GPU
    "ml.g4dn.8xlarge": 2.72,     # $ per hour - 1 T4 GPU
    "ml.g4dn.12xlarge": 4.08,    # $ per hour - 4 T4 GPUs
    "ml.g4dn.16xlarge": 5.44,    # $ per hour - 1 T4 GPU
    
    # Amazon Silicon options
    "ml.inf2.xlarge": 0.76,      # $ per hour - 1 Inferentia2 chip
    "ml.inf2.8xlarge": 2.85,     # $ per hour - 1 Inferentia2 chip
    "ml.inf2.24xlarge": 8.55,    # $ per hour - 6 Inferentia2 chips
    "ml.inf2.48xlarge": 17.1,    # $ per hour - 12 Inferentia2 chips
    "ml.trn1.2xlarge": 1.34,     # $ per hour - 1 Trainium chip
    "ml.trn1.32xlarge": 21.5,    # $ per hour - 16 Trainium chips
    "ml.g5g.xlarge": 0.79,       # $ per hour - 1 T4G GPU (Graviton)
    "ml.g5g.2xlarge": 1.02,      # $ per hour - 1 T4G GPU (Graviton)
    "ml.g5g.4xlarge": 1.46,      # $ per hour - 1 T4G GPU (Graviton)
    "ml.g5g.8xlarge": 2.33,      # $ per hour - 1 T4G GPU (Graviton)
    "ml.g5g.16xlarge": 4.66      # $ per hour - 2 T4G GPUs (Graviton)
}

# Storage costs
storage_costs = {
    "s3_standard": 0.023,        # $ per GB per month
    "ebs_gp3": 0.08,             # $ per GB per month
    "model_repository": 0.05     # $ per GB per month (approximate)
}

# Display instance costs
instance_df = pd.DataFrame({
    'Instance Type': list(instance_costs.keys()),
    'Cost per Hour ($)': list(instance_costs.values())
})

# Group by instance family
instance_df['Family'] = instance_df['Instance Type'].apply(lambda x: x.split('.')[1])
instance_df = instance_df.sort_values(['Family', 'Cost per Hour ($)'])

display(instance_df)

## 5. Define Optimization Impact Factors

Based on our experiments and industry benchmarks, we can define the typical impact of each optimization technique on model size, inference time, and instance requirements.

In [ ]:
# Define optimization impact factors
optimization_impacts = {
    "quantization": {
        "name": "Quantization (FP32 to INT8)",
        "size_reduction": 0.75,  # 75% reduction (4x smaller)
        "inference_speedup": 0.6,  # 40% faster
        "instance_downgrade": {
            "ml.p4d.24xlarge": "ml.g5.48xlarge",  # Downgrade from p4d to g5
            "ml.g5.12xlarge": "ml.g5.8xlarge",    # Downgrade within g5 family
            "ml.g5.2xlarge": "ml.g4dn.xlarge"     # Downgrade from g5 to g4dn
        },
        "implementation_cost": 5000,  # $ (engineering time + compute)
        "implementation_time_days": 5
    },
    "pruning": {
        "name": "Structured Pruning (30%)",
        "size_reduction": 0.7,  # 30% reduction
        "inference_speedup": 0.75,  # 25% faster
        "instance_downgrade": {
            "ml.p4d.24xlarge": "ml.g5.48xlarge",  # Downgrade from p4d to g5
            "ml.g5.12xlarge": "ml.g5.8xlarge",    # Downgrade within g5 family
            "ml.g5.2xlarge": "ml.g5.xlarge"       # Downgrade within g5 family
        },
        "implementation_cost": 8000,  # $ (engineering time + compute)
        "implementation_time_days": 10
    },
    "distillation": {
        "name": "Knowledge Distillation",
        "size_reduction": 0.5,  # 50% reduction
        "inference_speedup": 0.4,  # 60% faster
        "instance_downgrade": {
            "ml.p4d.24xlarge": "ml.g5.24xlarge",  # Significant downgrade
            "ml.g5.12xlarge": "ml.g5.4xlarge",    # Significant downgrade
            "ml.g5.2xlarge": "ml.g4dn.xlarge"     # Downgrade from g5 to g4dn
        },
        "implementation_cost": 20000,  # $ (engineering time + compute)
        "implementation_time_days": 20
    },
    "amazon_silicon": {
        "name": "Amazon Silicon (Inferentia2)",
        "size_reduction": 0.9,  # 10% reduction (model compilation)
        "inference_speedup": 0.3,  # 70% faster
        "instance_downgrade": {
            "ml.p4d.24xlarge": "ml.inf2.48xlarge",  # Switch to Inferentia
            "ml.g5.12xlarge": "ml.inf2.24xlarge",   # Switch to Inferentia
            "ml.g5.2xlarge": "ml.inf2.xlarge"       # Switch to Inferentia
        },
        "implementation_cost": 15000,  # $ (engineering time + compilation)
        "implementation_time_days": 15
    },
    "combined": {
        "name": "Combined Optimization (Quantization + Pruning + Inferentia2)",
        "size_reduction": 0.4,  # 60% reduction
        "inference_speedup": 0.2,  # 80% faster
        "instance_downgrade": {
            "ml.p4d.24xlarge": "ml.inf2.24xlarge",  # Major downgrade
            "ml.g5.12xlarge": "ml.inf2.8xlarge",    # Major downgrade
            "ml.g5.2xlarge": "ml.inf2.xlarge"       # Major downgrade
        },
        "implementation_cost": 30000,  # $ (engineering time + compute)
        "implementation_time_days": 30
    }
}

# Display optimization impacts
impact_df = pd.DataFrame({
    'Technique': [info['name'] for info in optimization_impacts.values()],
    'Size Reduction (%)': [info['size_reduction'] * 100 for info in optimization_impacts.values()],
    'Inference Speedup (%)': [(1 - info['inference_speedup']) * 100 for info in optimization_impacts.values()],
    'Implementation Cost ($)': [info['implementation_cost'] for info in optimization_impacts.values()],
    'Implementation Time (days)': [info['implementation_time_days'] for info in optimization_impacts.values()]
})

display(impact_df)

## 6. Calculate Cost Savings for Production Models

Let's calculate the cost savings for each production model scenario using different optimization techniques.

In [ ]:
# Define request volumes to analyze
request_volumes = {
    "low": 1000000,        # 1 million requests per month
    "medium": 10000000,    # 10 million requests per month
    "high": 100000000,     # 100 million requests per month
    "very_high": 1000000000  # 1 billion requests per month
}

# Function to calculate monthly inference cost
def calculate_monthly_inference_cost(model, optimization=None, request_volume=1000000):
    """Calculate monthly inference cost for a model with optional optimization."""
    # Get base model parameters
    size_gb = model["size_gb"]
    inference_time_ms = model["inference_time_ms"]
    instance_type = model["instance_type"]
    throughput = model["throughput"]
    
    # Apply optimization impacts if specified
    if optimization:
        impact = optimization_impacts[optimization]
        size_gb *= impact["size_reduction"]
        inference_time_ms *= impact["inference_speedup"]
        instance_type = impact["instance_downgrade"].get(instance_type, instance_type)
        throughput = throughput / impact["inference_speedup"]  # Increased throughput
    
    # Calculate instance hours needed
    requests_per_second = request_volume / (30 * 24 * 3600)  # Convert monthly to per second
    instances_needed = math.ceil(requests_per_second / throughput)
    instance_hours = instances_needed * 24 * 30  # Hours per month
    
    # Calculate costs
    instance_cost_per_hour = instance_costs.get(instance_type, 0)
    compute_cost = instance_hours * instance_cost_per_hour
    storage_cost = size_gb * storage_costs["model_repository"]
    
    return {
        "instance_type": instance_type,
        "instances_needed": instances_needed,
        "instance_hours": instance_hours,
        "compute_cost": compute_cost,
        "storage_cost": storage_cost,
        "total_cost": compute_cost + storage_cost
    }

# Calculate costs for each model, optimization technique, and request volume
cost_results = {}

for model_key, model in production_models.items():
    cost_results[model_key] = {}
    
    for volume_key, volume in request_volumes.items():
        cost_results[model_key][volume_key] = {}
        
        # Baseline (no optimization)
        baseline_cost = calculate_monthly_inference_cost(model, None, volume)
        cost_results[model_key][volume_key]["baseline"] = baseline_cost
        
        # Calculate costs for each optimization technique
        for opt_key in optimization_impacts.keys():
            opt_cost = calculate_monthly_inference_cost(model, opt_key, volume)
            cost_results[model_key][volume_key][opt_key] = opt_cost

# Display cost results for a specific model and request volume
model_key = "large_llm"
volume_key = "medium"

print(f"Monthly costs for {production_models[model_key]['name']} with {volume_key} request volume ({request_volumes[volume_key]:,} requests/month):")
print("\nBaseline (No Optimization):")
baseline = cost_results[model_key][volume_key]["baseline"]
print(f"  Instance Type: {baseline['instance_type']}")
print(f"  Instances Needed: {baseline['instances_needed']}")
print(f"  Compute Cost: ${baseline['compute_cost']:,.2f}")
print(f"  Storage Cost: ${baseline['storage_cost']:,.2f}")
print(f"  Total Cost: ${baseline['total_cost']:,.2f}")

for opt_key, impact in optimization_impacts.items():
    print(f"\n{impact['name']}:")
    opt = cost_results[model_key][volume_key][opt_key]
    print(f"  Instance Type: {opt['instance_type']}")
    print(f"  Instances Needed: {opt['instances_needed']}")
    print(f"  Compute Cost: ${opt['compute_cost']:,.2f}")
    print(f"  Storage Cost: ${opt['storage_cost']:,.2f}")
    print(f"  Total Cost: ${opt['total_cost']:,.2f}")
    print(f"  Monthly Savings: ${baseline['total_cost'] - opt['total_cost']:,.2f}")
    print(f"  Savings Percentage: {((baseline['total_cost'] - opt['total_cost']) / baseline['total_cost']) * 100:.2f}%")

## 7. Calculate ROI and Payback Period

Let's calculate the ROI and payback period for each optimization technique.

In [ ]:
# Function to calculate ROI and payback period
def calculate_roi(baseline_cost, optimized_cost, implementation_cost, months=12):
    """Calculate ROI and payback period for an optimization technique."""
    monthly_savings = baseline_cost - optimized_cost
    total_savings = monthly_savings * months
    roi = ((total_savings - implementation_cost) / implementation_cost) * 100 if implementation_cost > 0 else float('inf')
    payback_months = implementation_cost / monthly_savings if monthly_savings > 0 else float('inf')
    
    return {
        "monthly_savings": monthly_savings,
        "annual_savings": monthly_savings * 12,
        "roi_percent": roi,
        "payback_months": payback_months
    }

# Calculate ROI for each model, optimization technique, and request volume
roi_results = {}

for model_key, model in production_models.items():
    roi_results[model_key] = {}
    
    for volume_key, volume in request_volumes.items():
        roi_results[model_key][volume_key] = {}
        baseline_cost = cost_results[model_key][volume_key]["baseline"]["total_cost"]
        
        for opt_key, impact in optimization_impacts.items():
            optimized_cost = cost_results[model_key][volume_key][opt_key]["total_cost"]
            implementation_cost = impact["implementation_cost"]
            
            roi = calculate_roi(baseline_cost, optimized_cost, implementation_cost)
            roi_results[model_key][volume_key][opt_key] = roi

# Display ROI results for a specific model and request volume
model_key = "large_llm"
volume_key = "medium"

print(f"ROI Analysis for {production_models[model_key]['name']} with {volume_key} request volume ({request_volumes[volume_key]:,} requests/month):")

roi_data = []
for opt_key, impact in optimization_impacts.items():
    roi = roi_results[model_key][volume_key][opt_key]
    roi_data.append({
        "Technique": impact["name"],
        "Implementation Cost ($)": f"${impact['implementation_cost']:,.2f}",
        "Monthly Savings ($)": f"${roi['monthly_savings']:,.2f}",
        "Annual Savings ($)": f"${roi['annual_savings']:,.2f}",
        "ROI (1 year)": f"{roi['roi_percent']:.2f}%",
        "Payback Period (months)": f"{roi['payback_months']:.2f}"
    })

roi_df = pd.DataFrame(roi_data)
display(roi_df)

## 8. Visualize Cost Savings Across Request Volumes

Let's visualize how cost savings scale with request volume for different optimization techniques.

In [ ]:
# Prepare data for visualization
model_key = "large_llm"
viz_data = []

for volume_key, volume in request_volumes.items():
    baseline_cost = cost_results[model_key][volume_key]["baseline"]["total_cost"]
    
    # Add baseline data point
    viz_data.append({
        "Request Volume": volume_key,
        "Requests per Month": volume,
        "Optimization Technique": "Baseline (No Optimization)",
        "Monthly Cost ($)": baseline_cost,
        "Cost Reduction (%)": 0
    })
    
    # Add data points for each optimization technique
    for opt_key, impact in optimization_impacts.items():
        optimized_cost = cost_results[model_key][volume_key][opt_key]["total_cost"]
        cost_reduction_percent = ((baseline_cost - optimized_cost) / baseline_cost) * 100
        
        viz_data.append({
            "Request Volume": volume_key,
            "Requests per Month": volume,
            "Optimization Technique": impact["name"],
            "Monthly Cost ($)": optimized_cost,
            "Cost Reduction (%)": cost_reduction_percent
        })

# Create DataFrame for visualization
viz_df = pd.DataFrame(viz_data)

# Plot monthly costs across request volumes
plt.figure(figsize=(14, 8))
sns.barplot(x="Request Volume", y="Monthly Cost ($)", hue="Optimization Technique", data=viz_df)
plt.title(f"Monthly Costs by Request Volume for {production_models[model_key]['name']}")
plt.xlabel("Request Volume")
plt.ylabel("Monthly Cost ($)")
plt.yscale("log")  # Use log scale for better visualization
plt.xticks(rotation=0)
plt.legend(title="Optimization Technique", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Plot cost reduction percentage
plt.figure(figsize=(14, 8))
# Filter out baseline which has 0% reduction
reduction_df = viz_df[viz_df["Optimization Technique"] != "Baseline (No Optimization)"]
sns.barplot(x="Request Volume", y="Cost Reduction (%)", hue="Optimization Technique", data=reduction_df)
plt.title(f"Cost Reduction Percentage by Request Volume for {production_models[model_key]['name']}")
plt.xlabel("Request Volume")
plt.ylabel("Cost Reduction (%)")
plt.xticks(rotation=0)
plt.legend(title="Optimization Technique", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 9. Amazon Silicon Cost-Benefit Analysis

Let's take a closer look at the cost benefits of using Amazon Silicon (Inferentia2, Trainium, and Graviton) for model inference.

In [ ]:
# Define Amazon Silicon options and their benefits
amazon_silicon = {
    "inferentia2": {
        "name": "AWS Inferentia2",
        "description": "Custom silicon designed for high-performance, cost-effective ML inference",
        "best_for": "Production inference for transformer models, including LLMs",
        "cost_savings": "Up to 50% cost savings compared to GPU instances",
        "performance": "Up to 4x higher throughput than comparable GPU instances for transformer models",
        "instance_types": ["ml.inf2.xlarge", "ml.inf2.8xlarge", "ml.inf2.24xlarge", "ml.inf2.48xlarge"],
        "models": "Transformer models (BERT, GPT, T5, etc.)",
        "limitations": "Requires model compilation with AWS Neuron SDK"
    },
    "trainium": {
        "name": "AWS Trainium",
        "description": "Custom silicon designed for high-performance, cost-effective ML training",
        "best_for": "Training large language models and other deep learning models",
        "cost_savings": "Up to 40% cost savings compared to GPU instances for training",
        "performance": "Optimized for transformer model training",
        "instance_types": ["ml.trn1.2xlarge", "ml.trn1.32xlarge"],
        "models": "Transformer models (BERT, GPT, T5, etc.)",
        "limitations": "Requires model compilation with AWS Neuron SDK"
    },
    "graviton": {
        "name": "AWS Graviton",
        "description": "ARM-based processors designed for better price-performance",
        "best_for": "General-purpose computing and inference for smaller models",
        "cost_savings": "Up to 20% cost savings compared to x86-based instances",
        "performance": "Better price-performance ratio for many workloads",
        "instance_types": ["ml.g5g.xlarge", "ml.g5g.2xlarge", "ml.g5g.4xlarge", "ml.g5g.8xlarge", "ml.g5g.16xlarge"],
        "models": "Wide range of models with ARM-compatible frameworks",
        "limitations": "Requires ARM-compatible code and dependencies"
    }
}

# Display Amazon Silicon options
for silicon_key, silicon in amazon_silicon.items():
    print(f"\n{silicon['name']}")
    print(f"Description: {silicon['description']}")
    print(f"Best for: {silicon['best_for']}")
    print(f"Cost savings: {silicon['cost_savings']}")
    print(f"Performance: {silicon['performance']}")
    print(f"Instance types: {', '.join(silicon['instance_types'])}")
    print(f"Supported models: {silicon['models']}")
    print(f"Limitations: {silicon['limitations']}")

## 10. Conclusion and Recommendations

Based on our cost analysis, we can make the following recommendations for optimizing model deployment costs:

In [ ]:
# Display recommendations based on model size and request volume
recommendations = {
    "large_models": [
        "Use a combination of quantization, pruning, and Amazon Silicon for maximum cost savings",
        "Consider knowledge distillation for models with very high request volumes",
        "Use AWS Inferentia2 instances for inference to reduce costs by up to 50%",
        "Apply INT8 quantization to reduce model size by 75% with minimal accuracy impact",
        "Use structured pruning to further reduce model size and improve inference speed"
    ],
    "medium_models": [
        "Start with quantization as it offers the best ROI for medium-sized models",
        "Consider AWS Inferentia2 instances for high-volume inference workloads",
        "Use knowledge distillation if you need significant size reduction",
        "Apply structured pruning to improve inference speed",
        "Consider Graviton-based instances for cost-effective inference"
    ],
    "small_models": [
        "Focus on quantization as it offers the best ROI for small models",
        "Use Graviton-based instances for cost-effective inference",
        "Consider batch processing to improve throughput",
        "Apply simple optimizations like operator fusion and graph optimization",
        "Use auto-scaling to match capacity with demand"
    ],
    "general": [
        "Always benchmark optimized models to ensure they meet accuracy requirements",
        "Consider the trade-off between implementation cost and long-term savings",
        "Start with the optimization technique that offers the best ROI for your specific use case",
        "Combine multiple optimization techniques for maximum cost savings",
        "Monitor inference costs and performance to identify opportunities for further optimization"
    ]
}

# Display recommendations
print("Recommendations for Large Models (>10GB):")
for rec in recommendations["large_models"]:
    print(f"- {rec}")

print("\nRecommendations for Medium Models (1-10GB):")
for rec in recommendations["medium_models"]:
    print(f"- {rec}")

print("\nRecommendations for Small Models (<1GB):")
for rec in recommendations["small_models"]:
    print(f"- {rec}")

print("\nGeneral Recommendations:")
for rec in recommendations["general"]:
    print(f"- {rec}")